In [0]:
from pyspark.sql.functions import current_timestamp, input_file_name, col

CATALOG = "workspace"
LANDING = f"/Volumes/{CATALOG}/default/data/retail"
CKPT    = f"/Volumes/{CATALOG}/default/data/checkpoints/medallion/bronze"
CKPT_SCHEMA = f"/Volumes/{CATALOG}/default/data/checkpoints/medallion/schema_bronze"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")

options_common = {
  "cloudFiles.inferColumnTypes": "true",
  "cloudFiles.schemaEvolutionMode": "addNewColumns"
}

def ingest(entity, fmt="parquet", extra_opts=None):
    src = f"{LANDING}/{entity}/parquet-raw"
    ck  = f"{CKPT}/{entity}"

    (spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", fmt)
        .option("cloudFiles.schemaLocation", f"{CKPT_SCHEMA}/{entity}")
        .load(src)
        .withColumn("_ingest_ts", current_timestamp())
        .writeStream
        .option("checkpointLocation", ck)
        .trigger(availableNow=True)
        .table(f"{CATALOG}.bronze.{entity}"))


for t in ["customers","departments","categories","products","orders","order_items"]:
    ingest(t, fmt="parquet")